# Guide Evaluation Analysis

Analyzes JSON output from `guide-eval` to evaluate distance metrics as predictors for equality saturation reachability.

Run with: `uv run jupyter notebook guide_eval.ipynb`

In [ ]:
from pathlib import Path

import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from helpers import compute_correlations, load_guide_eval, plot_predictor_vs_iters

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 8)})

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────
DATA_DIR = Path("../data/guide_eval")
candidates = sorted(
    [f for f in DATA_DIR.iterdir() if f.is_dir()],
    key=lambda p: p.stat().st_mtime,
)
if not candidates:
    raise FileNotFoundError(f"No run-*.csv found in {DATA_DIR.resolve()}")
csv_file = candidates[-1] / "out.csv"

print(f"Using: {csv_file}")

df = load_guide_eval(csv_file)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# ── Overview ───────────────────────────────────────────────────────────
df.schema

In [ ]:
# ── Predictor columns ─────────────────────────────────────────────────
# All numeric columns that could predict iteration count.
# Extend this list if you add new distance metrics.
PREDICTOR_CANDIDATES = [
    # "zs_rank",
    # "structural_rank",
    "zs_distance",
    "structural_overlap",
    "structural_zs_sum",
]
TARGET = "iterations_to_reach"

# Keep only columns that actually exist in this CSV
predictors = [c for c in PREDICTOR_CANDIDATES if c in df.columns]
print(f"Predictors found: {predictors}")

## Per-goal analysis

In [ ]:
goals = df["goal"].unique().to_list()
print(f"{len(goals)} goal(s) in dataset")

In [ ]:
# ── Reachability summary per goal ──────────────────────────────────────
summary = (
    df.group_by("goal")
    .agg(
        pl.len().alias("total"),
        pl.col("reached").sum().alias("reached"),
        (pl.col("reached").mean() * 100).round(1).alias("reached_%"),
        pl.col("iterations_to_reach").filter(pl.col("reached")).min().alias("iter_min"),
        pl.col("iterations_to_reach")
        .filter(pl.col("reached"))
        .median()
        .alias("iter_median"),
        pl.col("iterations_to_reach").filter(pl.col("reached")).max().alias("iter_max"),
    )
    .with_columns(
        pl.when(pl.col("goal").str.len_chars() > 80)
        .then(pl.col("goal").str.slice(0, 80) + "...")
        .otherwise(pl.col("goal"))
        .alias("goal")
    )
    .sort("goal")
)
summary

## Correlation analysis

For each predictor, compute Spearman and Kendall-tau correlation with `iterations_to_reach` (reached guides only).

In [ ]:
# ── Per-goal correlations ──────────────────────────────────────────────
from IPython.display import display, Markdown

for goal in goals:
    g = df.filter(pl.col("goal") == goal)
    n_reached = len(g.filter(pl.col("reached")))
    display(Markdown(f"### Goal (n_reached={n_reached}): `{goal[:100]}`"))
    corr = compute_correlations(g, predictors, TARGET)
    if len(corr) > 0:
        display(corr)
    else:
        display(Markdown("*Not enough reached guides for correlation.*"))

## Scatter plots: predictor vs iterations to reach

In [ ]:
# ── All predictors, all goals pooled ───────────────────────────────────
n_pred = len(predictors)
cols = min(n_pred, 3)
rows = (n_pred + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows), squeeze=False)

for i, pred in enumerate(predictors):
    ax = axes[i // cols][i % cols]
    plot_predictor_vs_iters(df, pred, ax=ax, title_suffix=" (all goals)")

# Hide unused axes
for j in range(n_pred, rows * cols):
    axes[j // cols][j % cols].set_visible(False)

fig.tight_layout()
plt.show()

In [ ]:
# ── Per-goal scatter plots ─────────────────────────────────────────────
for goal in goals:
    g = df.filter(pl.col("goal") == goal)
    n_reached = len(g.filter(pl.col("reached")))
    if n_reached < 3:
        print(f"Skipping goal (only {n_reached} reached): {goal[:80]}")
        continue

    n_pred = len(predictors)
    cols = min(n_pred, 3)
    rows = (n_pred + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows), squeeze=False)
    for i, pred in enumerate(predictors):
        plot_predictor_vs_iters(
            g, pred, ax=axes[i // cols][i % cols], title_suffix=f"\ngoal: {goal[:60]}"
        )
    for j in range(n_pred, rows * cols):
        axes[j // cols][j % cols].set_visible(False)
    fig.tight_layout()
    plt.show()

In [ ]:
# ── Box plots: predictor values for reached vs not reached ──────────
# Subsample for box plots if data is large (box stats are robust to sampling)
BOX_MAX_N = 100_000

reached_df = df.filter(pl.col("reached"))
unreached_df = df.filter(~pl.col("reached"))

if len(reached_df) > BOX_MAX_N:
    reached_df = reached_df.sample(BOX_MAX_N, seed=42)
if len(unreached_df) > BOX_MAX_N:
    unreached_df = unreached_df.sample(BOX_MAX_N, seed=42)

n_pred = len(predictors)
cols = min(n_pred, 3)
rows = (n_pred + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows), squeeze=False)

for i, pred in enumerate(predictors):
    ax = axes[i // cols][i % cols]
    reached_vals = reached_df[pred].drop_nulls().to_numpy()
    unreached_vals = unreached_df[pred].drop_nulls().to_numpy()
    ax.boxplot(
        [reached_vals, unreached_vals],
        tick_labels=["reached", "not reached"],
        patch_artist=True,
        boxprops=dict(facecolor="lightblue"),
    )
    ax.set_title(pred)
    ax.set_ylabel(pred)

for j in range(n_pred, rows * cols):
    axes[j // cols][j % cols].set_visible(False)

fig.suptitle("Predictor distributions: reached vs not reached", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# ── Histograms of iterations_to_reach (reached only) ──────────────────
reached_all = df.filter(pl.col("reached"))
iters_vals = reached_all[TARGET].drop_nulls().to_numpy()

if len(iters_vals) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(
        iters_vals,
        bins=max(1, int(iters_vals.max() - iters_vals.min() + 1)),
        color="steelblue",
        edgecolor="white",
    )
    ax.set_xlabel("iterations_to_reach")
    ax.set_ylabel("count")
    ax.set_title("Distribution of iterations to reach goal")
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    fig.tight_layout()
    plt.show()
else:
    print("No reached guides to plot.")

## Iteration breakdown table

In [ ]:
# ── Breakdown by iterations to reach ──────────────────────────────────
# Reached: group by goal + iteration count
reached_breakdown = (
    df.filter(pl.col("reached"))
    .group_by("goal", TARGET)
    .agg(pl.len().alias("count"))
    .rename({TARGET: "iterations"})
)

# Unreached: count per goal
unreached_breakdown = (
    df.filter(~pl.col("reached"))
    .group_by("goal")
    .agg(pl.len().alias("count"))
    .with_columns(pl.lit(None, dtype=pl.Int64).alias("iterations"))
    .select("goal", "iterations", "count")
)

breakdown = (
    pl.concat([reached_breakdown, unreached_breakdown])
    .with_columns(
        pl.when(pl.col("goal").str.len_chars() > 60)
        .then(pl.col("goal").str.slice(0, 60) + "...")
        .otherwise(pl.col("goal"))
        .alias("goal")
    )
    .sort("goal", "iterations", nulls_last=True)
    .select("goal", "iterations", "count")
)
breakdown

## Predictor comparison heatmap

In [ ]:
# ── Spearman correlation heatmap across all predictors ─────────────────
from scipy.stats import spearmanr

reached = df.filter(pl.col("reached")).drop_nulls(subset=[TARGET])
cols_for_heatmap = [p for p in predictors if p in reached.columns] + [TARGET]

# Subsample for spearmanr to keep it fast
HEATMAP_MAX_N = 100_000
if len(reached) > HEATMAP_MAX_N:
    sample = reached.sample(HEATMAP_MAX_N, seed=42)
else:
    sample = reached

mat = sample.select(cols_for_heatmap).to_numpy().astype(float)
corr_matrix, _ = spearmanr(mat)  # returns full matrix directly

n_cols = len(cols_for_heatmap)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1)  # pyright: ignore[reportArgumentType]
ax.set_xticks(range(n_cols))
ax.set_yticks(range(n_cols))
ax.set_xticklabels(cols_for_heatmap, rotation=45, ha="right")
ax.set_yticklabels(cols_for_heatmap)

for i in range(n_cols):
    for j in range(n_cols):
        ax.text(
            j,
            i,
            f"{corr_matrix[i, j]:.2f}",  # pyright: ignore[reportIndexIssue]
            ha="center",
            va="center",
            fontsize=9,
            color="white" if abs(corr_matrix[i, j]) > 0.6 else "black",  # pyright: ignore[reportIndexIssue]
        )

fig.colorbar(im, ax=ax, label="Spearman ρ")
n_used = len(sample)
ax.set_title(f"Spearman correlation matrix (n={n_used:,})")
fig.tight_layout()
plt.show()